# 03 基差分析

四个品种的期货基差、期权隐含基差，并用 02 落库的**预测分红**修正。

- **读** `basis.db`：指数、期货、ETF、期权、SHIBOR、股息点指数、`dividend_forecast_daily`
- **写** `basis_daily`（一行一天一品种，列名 `{位次}_{指标}`），以及 `results/` 下的 HTML 报告和静态图

主要指标：`basis / basis_exdiv / basis_exdiv_pred`（期货基差：含分红 / 除已实现 / 除预测）、
`basis_opt*`（ETF 期权隐含）、`basis_iopt*`（股指期权隐含）、`*_minus_fut`（期权 − 期货）、`ann*`（年化 %）。
带 `_pred` 的列只用当日可知信息，可以回测；用了已实现分红的列含前视偏差。

In [ ]:
import os, re, json, sqlite3
from datetime import datetime

import numpy as np
import pandas as pd

ROOT = os.getcwd()
DB_PATH = os.path.join(ROOT, "basis.db")
RESULTS_DIR = os.path.join(ROOT, "results")

PRODUCTS = {
    "50":   {"index": "000016.SH", "fut": "IH", "divpt": "h50060.CSI", "etf": "510050.SH", "idx_opt": "HO", "label": "上证50"},
    "300":  {"index": "000300.SH", "fut": "IF", "divpt": "h30401.CSI", "etf": "510300.SH", "idx_opt": "IO", "label": "沪深300"},
    "500":  {"index": "000905.SH", "fut": "IC", "divpt": "h30402.CSI", "etf": "510500.SH", "idx_opt": None, "label": "中证500"},
    "1000": {"index": "000852.SH", "fut": "IM", "divpt": None,         "etf": None,        "idx_opt": "MO", "label": "中证1000"},
}

SLOTS = ["00", "01", "02", "03"]


SLOT_LABEL = {"00": "当月", "01": "次月", "02": "季月1", "03": "季月2"}

# ---------------------------------------------------------------- 基差计算参数
DAYS_PER_YEAR = 365


CONTRACT_MULTIPLIER = 10000


TRADEABLE_RATIO = 0.05


N_STRIKES_SIDE = 5


ETF_JUMP_MIN, ETF_JUMP_MAX = 0.4, 5.0


ETF_KEEP = 0.7


ETF_SNR = 2.0


ETF_GAP = 15


ETF_DIST_NOTICE_DAYS = 5

START, END = None, None    # 计算区间，None = 全部
PRODUCT, SLOT = "300", "01"

conn = sqlite3.connect(DB_PATH)

## 读库

In [ ]:
def _q(conn, sql, params, dates=("date",)):
    df = pd.read_sql(sql, conn, params=params)
    for c in dates:
        if c in df.columns and len(df):
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


def _rng(sql, p, start, end):
    if start:
        sql += " AND date>=?"; p.append(start)
    if end:
        sql += " AND date<=?"; p.append(end)
    return sql + " ORDER BY date", p


def load_index(conn, product, start=None, end=None):
    return _q(conn, *_rng("SELECT date, close AS index_close FROM index_daily WHERE product=?",
                          [product], start, end))


def load_series(conn, table, product, col="close", start=None, end=None):
    return _q(conn, *_rng(f"SELECT date, {col} FROM {table} WHERE product=?",
                          [product], start, end))


def load_shibor(conn, tenor="shibor_3m", start=None, end=None):
    return _q(conn, *_rng("SELECT date, value AS shibor FROM shibor WHERE tenor=?",
                          [tenor], start, end))


def load_options(conn, product, start=None, end=None):
    return _q(conn, *_rng("SELECT date, option_code, contract_name, close, volume, notional"
                          " FROM option_quotes WHERE product=?", [product], start, end))


def load_index_options(conn, product, start=None, end=None):
    return _q(conn, *_rng("SELECT date, option_code, contract_month, opt_type, strike,"
                          " close, volume, turnover FROM index_option_quotes WHERE product=?",
                          [product], start, end))


def load_futures_wide(conn, product, start=None, end=None):
    """四个合约位次摊平成一行一天，列名如 close_00 / days_00"""
    df = _q(conn, *_rng("SELECT * FROM futures_daily WHERE product=?", [product], start, end),
            dates=("date", "last_trading_date"))
    if df.empty:
        return df
    df["slot"] = df["contract"].str[-2:]
    out = None
    keep = ["contract_code", "close", "settle", "open_interest", "volume",
            "turnover", "last_trading_date", "days", "div_points"]
    for slot, g in df.groupby("slot"):
        g = g.set_index("date")[keep]
        g.columns = [f"{c}_{slot}" for c in g.columns]
        out = g if out is None else out.join(g, how="outer")
    return out.reset_index()


def load_forecast_wide(conn, product):
    """dividend_forecast_daily → 一行一天，列名 {slot}_div_pred（期货窗口）/ {slot}_div_pred_etf（ETF期权窗口）"""
    f = pd.read_sql("SELECT date, slot, pred_points FROM dividend_forecast_daily"
                    " WHERE product=?", conn, params=[product])
    if f.empty:
        return f
    # 库里 slot 存的是 'IF00' / 'IF00E' 这种写法，剥成 '00_div_pred' / '00_div_pred_etf'
    m = f["slot"].astype(str).str.extract(r"(\d{2})(E?)$")
    f["col"] = m[0] + "_div_pred" + m[1].map({"E": "_etf", "": ""})
    f = f.dropna(subset=["col"])
    wide = f.pivot(index="date", columns="col", values="pred_points")
    wide.columns = list(wide.columns)
    wide = wide.reset_index()
    wide["date"] = pd.to_datetime(wide["date"], errors="coerce")
    return wide

## 一、期权链

购沽配对；ETF 期权到期日 = 最后一个有行情的日子（仍在交易的按第 4 个周三），股指期权按到期月。

In [ ]:
def parse_etf_contract(name):
    """ETF期权中文合约名 → (购沽, 年, 月, 行权价, 是否调整合约)"""
    s = str(name).strip()
    if not s or s == "nan":
        return None
    typ = "C" if "购" in s else ("P" if "沽" in s else None)
    if typ is None:
        return None
    m = re.search(r"(\d{4})\s*年\s*(\d{1,2})\s*月", s)
    if m:
        yy, mm, tail = int(m.group(1)), int(m.group(2)), s[m.end():]
    else:
        m = re.search(r"(\d{1,2})\s*月", s)
        if not m:
            return None
        yy, mm, tail = None, int(m.group(1)), s[m.end():]
    t = re.search(r"(\d+)\s*([A-Z]?)\s*$", tail)
    if not t:
        return None
    raw = t.group(1)
    strike = float(raw) / 1000 if len(raw) >= 4 else float(raw)
    return typ, yy, mm, strike, bool(t.group(2))


def nth_weekday(y, m, weekday, n):
    """某月第 n 个星期几（weekday: 周一=0）。ETF期权=第4个周三，股指期权=第3个周五"""
    days = [d for d in pd.date_range(pd.Timestamp(y, m, 1), periods=31, freq="D")
            if d.weekday() == weekday and d.month == m]
    return days[n - 1] if len(days) >= n else None


def etf_option_expiry(y, m, cal=None):
    """ETF期权到期日：到期月第4个周三，遇非交易日顺延。cal 为交易日 DatetimeIndex，超出其范围则不顺延"""
    d = nth_weekday(y, m, 2, 4)
    if d is not None and cal is not None and len(cal) and d <= cal[-1]:
        d = cal[cal.searchsorted(d)]
    return d


def _pair_calls_puts(d, keys, vol_cols):
    ren = lambda t: {"close": f"{t}_close", vol_cols[0]: f"{t}_volume", vol_cols[1]: f"{t}_notional"}
    cols = keys + ["close", *vol_cols]
    call = d[d["typ"] == "C"][cols].rename(columns=ren("call")).drop_duplicates(keys)
    put = d[d["typ"] == "P"][cols].rename(columns=ren("put")).drop_duplicates(keys)
    return call.merge(put, on=keys, how="inner")


def build_etf_chain(q, verbose=True):
    """ETF期权逐合约行情 → 按 (日期, 到期日, 行权价, 是否调整) 配对的购沽链"""
    if q is None or q.empty:
        return pd.DataFrame()
    d = q.copy()
    d["date"] = pd.to_datetime(d["date"])
    for c in ("close", "volume", "notional"):
        d[c] = pd.to_numeric(d.get(c), errors="coerce")
    d = d.dropna(subset=["close"])
    d = d[d["close"] > 0]
    if d.empty:
        return pd.DataFrame()

    # 成交额单位识别：成交额/成交量/价格 应≈合约乘数，远小于它说明是万元
    m = d[(d["volume"] > 0) & (d["notional"] > 0)]
    if len(m) >= 100:
        imp = (m["notional"] / m["volume"] / m["close"]).median()
        if np.isfinite(imp) and imp < 100:
            d["notional"] *= CONTRACT_MULTIPLIER
            if verbose:
                print(f"    ℹ 成交额判定为万元(比值={imp:.2f})，已×{CONTRACT_MULTIPLIER}")

    info = d["contract_name"].apply(parse_etf_contract)
    d = d[info.notna()].copy()
    info = info[info.notna()]
    if d.empty:
        return pd.DataFrame()
    d["typ"], d["yy"], d["mm"], d["strike"], d["adj"] = zip(*info)

    # 到期日 = 该合约最后一个有行情的日子；仍在交易的合约按第4个周三推算
    d["expiry"] = d["option_code"].map(d.groupby("option_code")["date"].max())
    need = d["expiry"] >= d["date"].max() - pd.Timedelta(days=3)
    if need.any():
        fb = []
        for _, r in d[need].iterrows():
            if pd.isna(r["mm"]):
                fb.append(r["expiry"]); continue
            mm = int(r["mm"])
            yy = int(r["yy"]) if pd.notna(r["yy"]) else r["date"].year + (1 if mm < r["date"].month else 0)
            w = nth_weekday(yy, mm, 2, 4)
            fb.append(w if w is not None else r["expiry"])
        d.loc[need, "expiry"] = fb

    ch = _pair_calls_puts(d, ["date", "expiry", "strike", "adj"], ("volume", "notional"))
    ch["contract_month"] = pd.to_datetime(ch["expiry"]).dt.strftime("%y%m")
    ch = ch.sort_values(["date", "expiry", "strike"]).reset_index(drop=True)
    if verbose and len(ch):
        print(f"    ✓ ETF期权链 {len(ch)} 组配对，{ch['date'].nunique()} 个交易日")
    return ch


def build_index_chain(q, verbose=True):
    """股指期权逐合约行情 → 按 (日期, 到期月, 行权价) 配对。到期日=每月第3个周五"""
    if q is None or q.empty:
        return pd.DataFrame()
    d = q.copy()
    d["date"] = pd.to_datetime(d["date"])
    d["close"] = pd.to_numeric(d["close"], errors="coerce")
    d["strike"] = pd.to_numeric(d["strike"], errors="coerce")
    d = d.dropna(subset=["close", "strike", "opt_type", "contract_month"])
    d = d[d["close"] > 0].rename(columns={"opt_type": "typ"})
    if d.empty:
        return pd.DataFrame()
    ch = _pair_calls_puts(d, ["date", "contract_month", "strike"], ("volume", "turnover"))
    if ch.empty:
        return ch
    exp = {cm: nth_weekday(2000 + int(cm[:2]), int(cm[2:]), 4, 3) for cm in ch["contract_month"].unique()}
    ch["expiry"] = ch["contract_month"].map(exp)
    ch = ch.dropna(subset=["expiry"]).sort_values(["date", "expiry", "strike"]).reset_index(drop=True)
    if verbose and len(ch):
        print(f"    ✓ 股指期权链 {len(ch)} 组配对，{ch['date'].nunique()} 个交易日")
    return ch

## 二、PCP 反推远期、ETF 自身除息、区间分红

In [ ]:
def implied_forward(sl, r, T, spot, call_col="call_close", put_col="put_close"):
    """PCP 反推远期：F = K + (C − P)·e^{rT}，取平值上下各 N 档的中位数"""
    if sl is None or len(sl) == 0 or not np.isfinite(spot) or spot <= 0:
        return np.nan, 0
    df = sl.dropna(subset=[call_col, put_col, "strike"])
    df = df[(df[call_col] > 0) & (df[put_col] > 0)]
    if df.empty:
        return np.nan, 0
    near = pd.concat([df[df["strike"] <= spot].nlargest(N_STRIKES_SIDE, "strike"),
                      df[df["strike"] > spot].nsmallest(N_STRIKES_SIDE, "strike")])
    if near.empty:
        return np.nan, 0
    fw = near["strike"] + (near[call_col] - near[put_col]) * np.exp(r * max(T, 1e-6))
    fw = fw[np.isfinite(fw)]
    return (float(fw.median()), int(len(fw))) if len(fw) else (np.nan, 0)


def detect_etf_dist(idx, etf, verbose=True):
    """用 指数/ETF 比值的阶跃识别 ETF 自身除息日与金额"""
    if idx is None or etf is None or idx.empty or etf.empty:
        return pd.DataFrame(columns=["ex_date", "amount"])
    m = idx.merge(etf.rename(columns={"close": "etf_close"}), on="date").sort_values("date").reset_index(drop=True)
    m["ratio"] = m["index_close"] / m["etf_close"]
    m["jump"] = m["ratio"].pct_change() * 100
    hits = []
    for k in m.index[(m["jump"] > ETF_JUMP_MIN) & (m["jump"] <= ETF_JUMP_MAX)]:
        j = m["jump"].iloc[k]
        b, a = m["ratio"].iloc[max(0, k - 6):k].median(), m["ratio"].iloc[k:k + 6].median()
        exp = b * j / 100
        if not np.isfinite(b) or not np.isfinite(a) or exp <= 0 or (a - b) / exp < ETF_KEEP:
            continue
        noise = m["jump"].iloc[max(0, k - 20):k].std()
        if np.isfinite(noise) and noise > 0 and j / noise < ETF_SNR:
            continue
        hits.append((k, m["date"].iloc[k], j, m["etf_close"].iloc[k]))
    hits.sort(key=lambda t: -t[2])
    kept = []
    for h in hits:
        if all(abs(h[0] - x[0]) >= ETF_GAP for x in kept):
            kept.append(h)
    kept.sort(key=lambda t: t[0])
    r = pd.DataFrame([{"ex_date": dt, "amount": px * (j / 100) / (1 + j / 100)} for _, dt, j, px in kept],
                     columns=["ex_date", "amount"])
    if verbose and len(r):
        print("    ℹ ETF除息 %d 次: %s" % (len(r), ", ".join(
            f"{pd.Timestamp(d).date()}({a:.3f})" for d, a in zip(r["ex_date"], r["amount"]))))
    return r


def dividend_cum(dpi):
    """股息点指数 → 累计已实现分红点数（跨年重设处接上），区间分红 = 两端差值"""
    if dpi is None or dpi.empty:
        return None
    d = dpi.sort_values("date").set_index("date")["value"].astype(float)
    inc = d.diff()
    reset = inc < -1                        # 每年12月重设归零：当天值就是归零后当天的分红
    inc[reset] = d[reset]
    return inc.fillna(0.0).cumsum()


def _window_div(cum, a, b):
    """(a, b] 区间已实现分红；b 超出股息点数据范围（尚未发生）时为 NaN"""
    if cum is None or b > cum.index[-1] or a < cum.index[0]:
        return np.nan
    return float(cum.asof(b) - cum.asof(a))

## 三、基差计算

期权按到期月对齐期货；ETF 期权的区间分红按期权自己的到期日；ETF 自身分红只在公告后加回。

In [ ]:
def _ann(x, S, dte):
    return x / S * DAYS_PER_YEAR / dte * 100


def _fut_month(code):
    """期货合约代码 → 到期年月 'YYMM'（IF2503 → '2503'），用来找同月到期的期权"""
    m = re.search(r"(\d{4})$", str(code))
    return m.group(1) if m else None


def _cont_rate(shibor, T):
    """SHIBOR(单利, 小数) → 连续复利"""
    return np.log(1 + shibor * T) / T if (np.isfinite(shibor) and T > 0 and 1 + shibor * T > 0) else 0.0


def compute(idx, fut, etf, sh, etf_chain, index_chain, dist, verbose=True, div_cum=None):
    """div_cum：累计已实现分红（dividend_cum），用于按 ETF 期权自己的到期日算区间分红"""
    out = idx.merge(fut, on="date").merge(sh, on="date", how="left")
    out = (out.merge(etf.rename(columns={"close": "etf_close"}), on="date", how="left")
           if not etf.empty else out.assign(etf_close=np.nan))
    out["shibor"] = out["shibor"].ffill()
    out["etf_to_index"] = out["index_close"] / out["etf_close"]

    ecbd = {d: g for d, g in etf_chain.groupby("date")} if len(etf_chain) else {}
    icbd = {d: g for d, g in index_chain.groupby("date")} if len(index_chain) else {}
    dist_dates = pd.to_datetime(dist["ex_date"]).reset_index(drop=True) if len(dist) else None
    # ETF 自身分红在 T 日是否已知：除息日前 ETF_DIST_NOTICE_DAYS 个交易日起才算公告了（point-in-time）
    if dist_dates is not None:
        tdays = pd.DatetimeIndex(sorted(out["date"].unique()))
        pos = tdays.searchsorted(dist_dates)
        dist_known = pd.Series([tdays[max(i - ETF_DIST_NOTICE_DAYS, 0)] for i in pos])
        dist_amt = dist["amount"].reset_index(drop=True)

    recs = []
    for _, row in out.iterrows():
        S, dt = row["index_close"], row["date"]
        if not np.isfinite(S) or S <= 0:
            continue
        rec = {"date": dt, "spot": S, "etf_close": row.get("etf_close"), "shibor": row.get("shibor")}
        ec, ic = ecbd.get(dt), icbd.get(dt)
        sb = row.get("shibor")

        # 期权按【到期月】对齐期货，不能按"第几个到期"：
        #   股指期权挂 当月+下2月+3个季月，期货挂 当月+次月+2个季月，序号在季月位次上错开；
        #   ETF期权第4个周三到期，期货第3个周五，两者之间那几天 ETF 当月还没到期、期货已换月。
        # 找不到同月期权的位次（如上述那几天的期货次月）留空，不拿别的月份凑。
        for s in SLOTS:
            F, ltd = row.get(f"close_{s}"), row.get(f"last_trading_date_{s}")
            ym = _fut_month(row.get(f"contract_code_{s}"))
            if not np.isfinite(F) or F <= 0:
                continue
            # 到期日当天用结算价（交割价口径）
            if pd.notna(ltd) and pd.Timestamp(ltd).normalize() == dt.normalize():
                st = row.get(f"settle_{s}")
                if np.isfinite(st) and st > 0:
                    F = st
            dte = row.get(f"days_{s}")
            if not np.isfinite(dte) or dte <= 0:
                dte = ((pd.Timestamp(ltd) - dt).days + 1) if pd.notna(ltd) else np.nan
            # div      = 事后已实现分红（股息点指数区间差值），含前视偏差，只能描述历史
            # div_pred = 当日可知信息下的预测分红（dividend_forecast_daily），可用于回测
            div = row.get(f"div_points_{s}")
            dpred = row.get(f"{s}_div_pred", np.nan)
            has_div, has_pred = np.isfinite(div), np.isfinite(dpred)
            basis = F - S
            rec.update({f"{s}_F": F, f"{s}_DTE": dte, f"{s}_div": div, f"{s}_basis": basis,
                        f"{s}_div_pred": dpred,
                        f"{s}_basis_exdiv": basis + div if has_div else np.nan,
                        f"{s}_basis_exdiv_pred": basis + dpred if has_pred else np.nan,
                        f"{s}_div_err": dpred - div if (has_pred and has_div) else np.nan})
            if np.isfinite(dte) and dte > 0:
                rec[f"{s}_ann"] = _ann(basis, S, dte)
                if has_div:
                    rec[f"{s}_ann_exdiv"] = _ann(basis + div, S, dte)
                if has_pred:
                    rec[f"{s}_ann_exdiv_pred"] = _ann(basis + dpred, S, dte)

            # ---- ETF期权：ETF 在成分股除息时不跌（现金留在基金里，到ETF自身除息才分），
            # 所以 ETF 远期换算成指数点位后是"不含分红"的，要减去区间分红才和期货同口径
            sl = ec[ec["contract_month"] == ym] if (ec is not None and ym) else None
            if sl is not None and len(sl) and np.isfinite(row.get("etf_close", np.nan)):
                oe = pd.Timestamp(sl["expiry"].max())
                sl = sl[sl["expiry"] == oe]
                odte = (oe - dt).days + 1
                T = odte / DAYS_PER_YEAR
                f_etf, _ = implied_forward(sl, _cont_rate(sb, T), T, row["etf_close"])
                if np.isfinite(f_etf):
                    # 期权窗口内的 ETF 自身分红要加回。只加 T 日已公告的：
                    # 还没公告的当日不可知，按 0 处理 —— 窗口跨 ETF 除息日时 ETF 版会偏低几十点，
                    # 直到公告日跳回，这是不用未来信息的代价
                    add = float(dist_amt[(dist_dates > dt) & (dist_dates <= oe) & (dist_known <= dt)].sum()) \
                        if dist_dates is not None else 0.0
                    # raw = 不含分红的指数远期基差，不依赖成分股分红 → 就是 exdiv 版
                    raw = (f_etf + add) * row["etf_to_index"] - S
                    # 区间分红按【期权自己的到期日】算：ETF期权比同月期货晚几天到期
                    div_o = _window_div(div_cum, dt, oe) if div_cum is not None else div
                    dpred_o = row.get(f"{s}_div_pred_etf", np.nan) if f"{s}_div_pred_etf" in row.index else dpred
                    has_div_o, has_pred_o = np.isfinite(div_o), np.isfinite(dpred_o)
                    rec.update({f"{s}_opt_DTE": odte, f"{s}_basis_opt_exdiv": raw,
                                f"{s}_opt_div": div_o, f"{s}_opt_div_pred": dpred_o})
                    # 含分红口径要减分红：已实现版含前视偏差，另出一份减【预测】分红的，可回测。
                    # 没有已实现分红（样本末端）就留空，不能悄悄当 0 减，否则同一列混两种口径
                    if has_div_o:
                        bo = raw - div_o
                        rec.update({f"{s}_basis_opt": bo, f"{s}_opt_minus_fut": bo - basis})
                    if has_pred_o:
                        rec.update({f"{s}_basis_opt_pred": raw - dpred_o,
                                    f"{s}_opt_minus_fut_pred": raw - dpred_o - basis})
                    # 年化统一口径：期权用期权自己的DTE，不是期货的
                    if odte > 0:
                        rec[f"{s}_ann_opt_exdiv"] = _ann(raw, S, odte)
                        if has_div_o:
                            rec[f"{s}_ann_opt"] = _ann(raw - div_o, S, odte)
                        if has_pred_o:
                            rec[f"{s}_ann_opt_pred"] = _ann(raw - dpred_o, S, odte)
                    for src, dst in [("volume", "tradeable_vol"), ("notional", "tradeable_notional")]:
                        t = sl.dropna(subset=[f"call_{src}", f"put_{src}"])
                        if len(t):
                            rec[f"{s}_{dst}"] = float(np.minimum(t[f"call_{src}"].values,
                                                                 t[f"put_{src}"].values).sum() * TRADEABLE_RATIO)

            # ---- 股指期权：标的就是价格指数，PCP 反推的远期【已经】反映了区间分红，
            # 与期货同口径，不能再减分红（旧版多减了一次，导致分红季 iopt−期货 ≈ −分红）
            sl = ic[ic["contract_month"] == ym] if (ic is not None and ym) else None
            if sl is not None and len(sl):
                # 股指期权与股指期货同为第3个周五到期、遇假顺延，直接用期货最后交易日，
                # 比自己推算的第3个周五更准（推算没处理节假日）
                oe = pd.Timestamp(ltd) if pd.notna(ltd) else pd.Timestamp(sl["expiry"].iloc[0])
                odte = (oe - dt).days + 1
                T = odte / DAYS_PER_YEAR
                f_idx, _ = implied_forward(sl, _cont_rate(sb, T), T, S)
                if np.isfinite(f_idx):
                    bi = f_idx - S
                    rec.update({f"{s}_iopt_DTE": odte, f"{s}_basis_iopt": bi,
                                f"{s}_iopt_minus_fut": bi - basis})
                    if odte > 0:
                        rec[f"{s}_ann_iopt"] = _ann(bi, S, odte)
                        if has_div:
                            rec[f"{s}_basis_iopt_exdiv"] = bi + div
                            rec[f"{s}_ann_iopt_exdiv"] = _ann(bi + div, S, odte)
                        if has_pred:
                            rec[f"{s}_basis_iopt_exdiv_pred"] = bi + dpred
                            rec[f"{s}_ann_iopt_exdiv_pred"] = _ann(bi + dpred, S, odte)
        recs.append(rec)

    res = pd.DataFrame(recs)
    if verbose and len(res):
        print(f"  ✓ {len(res)} 行 ({res['date'].min().date()} ~ {res['date'].max().date()})")
    return res

In [ ]:
def attach_forecast(conn, product, fut, verbose=True):
    """把 dividend_forecast_daily 拼成 {slot}_div_pred 列并入期货宽表。表为空时跳过"""
    wide = load_forecast_wide(conn, product)
    if wide.empty:
        if verbose:
            print("  · 无预测分红表，跳过（先跑 python -m basis precompute）")
        return fut
    out = fut.merge(wide, on="date", how="left")
    if verbose:
        got = [c for c in out.columns if c.endswith("_div_pred")]
        n = int(out[got].notna().any(axis=1).sum()) if got else 0
        print(f"  · 预测分红：{len(got)}个位次，{n}/{len(out)} 天有值")
    return out


def analyze_product(conn, product, start=None, end=None, verbose=True):
    """一个品种的完整流程：取数 → 建期权链 → 识别ETF除息 → 并入预测分红 → 算基差"""
    cfg = PRODUCTS[product]
    if verbose:
        print(f"[{cfg['label']}（{product}）]")
    idx = load_index(conn, product, start, end)
    fut = load_futures_wide(conn, product, start, end)
    if idx.empty or fut.empty:
        if verbose:
            print("  ✗ 无指数或期货数据，跳过")
        return pd.DataFrame()
    etf = load_series(conn, "etf_daily", product, "close", start, end)
    sh = load_shibor(conn, "shibor_3m", start, end)
    oq = load_options(conn, product, start, end)
    ioq = load_index_options(conn, product, start, end) if cfg.get("idx_opt") else pd.DataFrame()

    etf_chain = build_etf_chain(oq, verbose) if len(oq) else pd.DataFrame()
    index_chain = build_index_chain(ioq, verbose) if len(ioq) else pd.DataFrame()
    dist = detect_etf_dist(idx, etf, verbose) if len(etf) else pd.DataFrame(columns=["ex_date", "amount"])
    div_cum = dividend_cum(load_series(conn, "dividend_point_index", product, "value")) if len(oq) else None
    fut = attach_forecast(conn, product, fut, verbose)
    return compute(idx, fut, etf, sh, etf_chain, index_chain, dist, verbose, div_cum=div_cum)


def analyze_all(conn, start=None, end=None, products=None, verbose=True):
    results = {}
    for p in products or PRODUCTS:
        r = analyze_product(conn, p, start, end, verbose)
        if len(r):
            results[p] = r
        if verbose:
            print()
    return results

In [ ]:
results = analyze_all(conn, START, END)
basis_daily = pd.concat([r.assign(product=p) for p, r in results.items()], ignore_index=True)
basis_daily["date"] = basis_daily["date"].dt.strftime("%Y-%m-%d")
basis_daily.to_sql("basis_daily", conn, if_exists="replace", index=False)
print(f"→ basis_daily 写入 {len(basis_daily):,} 行 × {basis_daily.shape[1]} 列")

# ETF 自身分红（从「指数/ETF」比值阶跃识别），供 04 做公告天数的敏感性分析
etf_dist = pd.concat([detect_etf_dist(load_index(conn, p, START, END), load_series(conn, "etf_daily", p, "close", START, END),
                                      verbose=False).assign(product=p)
                      for p in PRODUCTS if PRODUCTS[p]["etf"]], ignore_index=True)
etf_dist["ex_date"] = pd.to_datetime(etf_dist["ex_date"]).dt.strftime("%Y-%m-%d")
etf_dist.to_sql("etf_dist_detected", conn, if_exists="replace", index=False)
print(f"→ etf_dist_detected 写入 {len(etf_dist)} 行")

## 四、统计表

In [ ]:
C_FUT, C_OPT, C_IOPT, C_DIV, C_PRED = "#2563eb", "#ea580c", "#0891b2", "#9333ea", "#7c3aed"


SLOT_C = {"00": "#2563eb", "01": "#f59e0b"}


HTML_SLOTS = ["00", "01"]


STAT_METRICS = [
    ("basis", "期货基差", "点"), ("basis_exdiv", "期货基差·除分红", "点"),
    ("basis_exdiv_pred", "期货基差·除预测分红", "点"),
    ("basis_opt", "ETF期权隐含", "点"), ("basis_opt_pred", "ETF期权隐含·减预测分红", "点"),
    ("basis_opt_exdiv", "ETF期权隐含·除分红", "点"),
    ("basis_iopt", "股指期权隐含", "点"), ("basis_iopt_exdiv", "股指期权隐含·除分红", "点"),
    ("basis_iopt_exdiv_pred", "股指期权隐含·除预测分红", "点"),
    ("ann", "期货年化", "%"), ("ann_exdiv", "期货年化·除分红", "%"), ("ann_exdiv_pred", "期货年化·除预测", "%"),
    ("ann_opt", "ETF期权年化", "%"), ("ann_opt_pred", "ETF期权年化·减预测分红", "%"),
    ("ann_opt_exdiv", "ETF期权年化·除分红", "%"),
    ("ann_iopt", "股指期权年化", "%"), ("ann_iopt_exdiv", "股指期权年化·除分红", "%"),
    ("ann_iopt_exdiv_pred", "股指期权年化·除预测", "%"),
    ("opt_minus_fut", "ETF期权−期货", "点"), ("opt_minus_fut_pred", "ETF期权−期货·预测分红", "点"),
    ("iopt_minus_fut", "股指期权−期货", "点"),
    ("div", "已实现分红", "点"), ("div_pred", "预测分红", "点"), ("div_err", "分红预测误差", "点"),
]


def summary_stats(res: pd.DataFrame, slots=SLOTS, metrics=STAT_METRICS) -> pd.DataFrame:
    """单品种各位次各指标的分布统计。当月年化受 DTE 放大影响，只看中位数"""
    rows = []
    for s in slots:
        for c, lab, unit in metrics:
            col = f"{s}_{c}"
            if col not in res.columns:
                continue
            v = pd.to_numeric(res[col], errors="coerce").dropna()
            if v.empty:
                continue
            rows.append({"位次": SLOT_LABEL[s], "指标": lab, "单位": unit, "样本": len(v),
                         "中位数": round(v.median(), 2), "均值": round(v.mean(), 2),
                         "标准差": round(v.std(), 2), "最小": round(v.min(), 2), "最大": round(v.max(), 2),
                         "p5": round(v.quantile(.05), 2), "p95": round(v.quantile(.95), 2)})
    return pd.DataFrame(rows)


def cross_product_table(results: dict, slot="00") -> pd.DataFrame:
    """跨品种对比（中位数口径）"""
    items = [("basis", "期货基差", "点"), ("basis_opt", "ETF期权隐含", "点"),
             ("basis_iopt", "股指期权隐含", "点"), ("opt_minus_fut", "ETF期权−期货", "点"),
             ("iopt_minus_fut", "股指期权−期货", "点"), ("div", "区间已实现分红", "点"),
             ("ann", "期货年化", "%"), ("ann_exdiv_pred", "期货年化·除预测分红", "%"),
             ("ann_opt", "ETF期权年化", "%"), ("ann_iopt", "股指期权年化", "%")]
    rows = []
    for c, lab, unit in items:
        row = {"指标": f"{SLOT_LABEL[slot]}{lab}", "单位": unit}
        for p, df in results.items():
            col = f"{slot}_{c}"
            v = pd.to_numeric(df[col], errors="coerce").dropna() if col in df.columns else pd.Series(dtype=float)
            row[PRODUCTS[p]["label"]] = round(float(v.median()), 2) if len(v) else None
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
cross_product_table(results, slot=SLOT)

In [ ]:
summary_stats(results[PRODUCT])   # 当月年化受剩余天数放大，只看中位数

## 五、单品种看图

实线 = 含分红，虚线 = 除已实现分红，点线 = 除预测分红（可回测）。改上面的 `PRODUCT` / `SLOT` 切换。

In [ ]:
def panels_for_slot(s):
    """单个位次的面板。含分红=实线，除分红=虚线，两者间距就是分红"""
    return [
        ("基差：含分红 vs 除分红", "点", [
            (f"{s}_basis", "期货基差", C_FUT, "solid"),
            (f"{s}_basis_exdiv", "期货基差·除分红", C_FUT, "dash"),
            (f"{s}_basis_exdiv_pred", "期货·除【预测】分红", C_PRED, "dot"),
            (f"{s}_basis_opt", "ETF期权隐含", C_OPT, "solid"),
            (f"{s}_basis_opt_pred", "ETF期权隐含·减【预测】分红", C_OPT, "dot"),
            (f"{s}_basis_opt_exdiv", "ETF期权隐含·除分红", C_OPT, "dash"),
            (f"{s}_basis_iopt", "股指期权隐含", C_IOPT, "dashdot"),
            (f"{s}_basis_iopt_exdiv", "股指期权隐含·除分红", C_IOPT, "dash"),
            (f"{s}_basis_iopt_exdiv_pred", "股指期权·除【预测】分红", C_IOPT, "dot"),
        ]),
        ("年化基差：含分红 vs 除分红", "%", [
            (f"{s}_ann", "期货年化", C_FUT, "solid"),
            (f"{s}_ann_exdiv", "期货年化·除分红", C_FUT, "dash"),
            (f"{s}_ann_exdiv_pred", "期货年化·除【预测】", C_PRED, "dot"),
            (f"{s}_ann_opt", "ETF期权年化", C_OPT, "solid"),
            (f"{s}_ann_opt_pred", "ETF期权年化·减【预测】", C_OPT, "dot"),
            (f"{s}_ann_opt_exdiv", "ETF期权年化·除分红", C_OPT, "dash"),
            (f"{s}_ann_iopt", "股指期权年化", C_IOPT, "dashdot"),
            (f"{s}_ann_iopt_exdiv", "股指期权年化·除分红", C_IOPT, "dash"),
            (f"{s}_ann_iopt_exdiv_pred", "股指期权年化·除【预测】", C_IOPT, "dot"),
        ]),
        ("分红：事后已实现 vs 当日可预测", "点", [
            (f"{s}_div", "已实现（事后真值）", C_DIV, "solid"),
            (f"{s}_div_pred", "预测（当日可知）", C_PRED, "dash"),
            (f"{s}_div_err", "预测−已实现", "#94a3b8", "dot")]),
        ("期权隐含 − 期货（定价偏离）", "点", [
            (f"{s}_opt_minus_fut", "ETF期权−期货", C_OPT, "solid"),
            (f"{s}_opt_minus_fut_pred", "ETF期权−期货·预测分红", C_OPT, "dot"),
            (f"{s}_iopt_minus_fut", "股指期权−期货", C_IOPT, "dashdot")]),
        ("期权可用流动性", "", [
            (f"{s}_tradeable_vol", "可用成交量(张)", "#0891b2", "solid"),
            (f"{s}_tradeable_notional", "可用成交额(元)", "#be123c", "solid|y2")]),
    ]


def panels_term():
    """期限对比：当月 vs 次月。年化那张已消除天数差异，可直接横比"""
    return [
        ("期货基差：当月 vs 次月", "点",
         [(f"{s}_basis", SLOT_LABEL[s], SLOT_C[s], "solid") for s in HTML_SLOTS]),
        ("期权隐含基差：当月 vs 次月", "点",
         [(f"{s}_basis_opt", SLOT_LABEL[s] + "·ETF版", SLOT_C[s], "solid") for s in HTML_SLOTS]
         + [(f"{s}_basis_iopt", SLOT_LABEL[s] + "·股指版", SLOT_C[s], "dashdot") for s in HTML_SLOTS]),
        ("年化基差期限结构", "%",
         [(f"{s}_ann", SLOT_LABEL[s] + "期货", SLOT_C[s], "solid") for s in HTML_SLOTS]
         + [(f"{s}_ann_opt", SLOT_LABEL[s] + "ETF期权", SLOT_C[s], "dash") for s in HTML_SLOTS]),
    ]


def _build(df, defs):
    out = []
    for title, unit, traces in defs:
        ts = []
        for col, name, color, dash in traces:
            if col not in df.columns:
                continue
            v = pd.to_numeric(df[col], errors="coerce")
            if v.notna().sum() == 0:
                continue
            ts.append({"name": name, "color": color, "dash": dash.replace("|y2", ""),
                       "y2": dash.endswith("|y2"),
                       "y": [None if pd.isna(x) else round(float(x), 4) for x in v]})
        if ts:
            out.append({"title": title, "unit": unit, "traces": ts})
    return out


def figure(res: pd.DataFrame, panel, height=380, title=None):
    """把一个面板定义画成 plotly 图，notebook 里 figure(...).show()"""
    import plotly.graph_objects as go
    ptitle, unit, _ = panel
    built = _build(res, [panel])
    fig = go.Figure()
    for t in (built[0]["traces"] if built else []):
        fig.add_trace(go.Scatter(x=res["date"], y=t["y"], name=t["name"], mode="lines",
                                 yaxis="y2" if t["y2"] else "y",
                                 line=dict(color=t["color"], width=1.6, dash=t["dash"])))
    fig.update_layout(height=height, title=title or f"{ptitle}" + (f"（{unit}）" if unit else ""),
                      hovermode="x unified", template="plotly_white",
                      margin=dict(l=60, r=60, t=50, b=40), legend=dict(orientation="h", y=-0.18))
    if any(t["y2"] for t in (built[0]["traces"] if built else [])):
        fig.update_layout(yaxis2=dict(overlaying="y", side="right", showgrid=False))
    fig.update_yaxes(zeroline=True, zerolinecolor="#94a3b8")
    return fig

In [ ]:
res = results[PRODUCT]
name = f"{PRODUCTS[PRODUCT]['label']} {SLOT_LABEL[SLOT]}"
for pnl in panels_for_slot(SLOT):
    figure(res, pnl, title=f"{name}：{pnl[0]}").show()
for pnl in panels_term():
    figure(res, pnl, title=f"{PRODUCTS[PRODUCT]['label']}：{pnl[0]}").show()

## 六、导出 HTML 报告

总览页 + 每品种一页，写到 `results/`，图表库走 CDN。

In [ ]:
_CSS = """
 body{font-family:-apple-system,"PingFang SC","Microsoft YaHei",sans-serif;
      margin:0;padding:0 24px 40px;background:#f8fafc;color:#0f172a}
 .nav{position:sticky;top:0;background:#f8fafcf2;backdrop-filter:blur(6px);
      padding:14px 0 10px;border-bottom:1px solid #e2e8f0;margin-bottom:18px;z-index:9}
 .nav a{display:inline-block;padding:6px 15px;margin:2px 6px 2px 0;border:1px solid #cbd5e1;
        border-radius:6px;background:#fff;color:#0f172a;text-decoration:none;font-size:13px}
 .nav a.on{background:#2563eb;color:#fff;border-color:#2563eb}
 .nav a.jump{border-style:dashed;color:#0f766e}
 h1{font-size:20px;margin:16px 0 4px}
 h2{font-size:16px;margin:26px 0 10px;padding-left:9px;border-left:4px solid #2563eb}
 .sub{color:#64748b;font-size:13px;line-height:1.6}
 .card{background:#fff;border:1px solid #e2e8f0;border-radius:8px;padding:16px;margin-bottom:18px;overflow-x:auto}
 .hint{font-size:12px;color:#64748b;margin-bottom:10px}
 .pnl{margin-bottom:22px}
 .ptitle{font-size:13px;font-weight:600;color:#334155;margin:0 0 2px 4px}
 table{border-collapse:collapse;font-size:13px;width:100%}
 th,td{padding:5px 10px;border-bottom:1px solid #e2e8f0;text-align:right}
 th:first-child,td:first-child,th:nth-child(2),td:nth-child(2){text-align:left}
 th{color:#475569;font-weight:600;background:#f1f5f9;position:sticky;top:52px}
 td.neg{color:#dc2626} td.pos{color:#1d4ed8} td.med{font-weight:600;background:#f8fafc}
 tr.sep td{border-top:2px solid #cbd5e1}
"""


_PAGE = """<!DOCTYPE html>
<html lang="zh-CN"><head><meta charset="utf-8"><title>__TITLE__</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>__CSS__</style></head><body>
<div class="nav">__NAV__</div>
<h1>__TITLE__</h1>
<div class="sub">生成于 __TS__ · 拖拽框选放大时间区间，双击还原 · 点击图例可显示/隐藏单条线<br>
实线=含分红，虚线=除分红（两者间距即区间已实现分红）；点划线=股指期权版<br>
<b>点线</b>=用【预测分红】修正，只用当日可知信息，可用于回测；其余分红线是事后已实现值，含前视偏差，只能描述历史<br>
年化口径：基差/指数 × 365/剩余天数；期货用期货剩余天数，期权用期权自己的剩余天数</div>
__SECTIONS__
<h2 id="stats">摘要统计（全区间）</h2>
<div class="card"><div class="hint">当月年化受剩余天数放大，只看中位数（已加粗）</div><div id="stats"></div></div>
<script>
const DATA = __DATA__;
DATA.views.forEach((v,vi)=>{
  const host=document.getElementById('chart'+vi);
  host.innerHTML=v.panels.map((p,i)=>
    `<div class="pnl"><div class="ptitle">${p.title}${p.unit?' ('+p.unit+')':''}</div>`
    +`<div id="p${vi}_${i}"></div></div>`).join('');
  v.panels.forEach((p,i)=>{
    const tr=p.traces.map(t=>({
      x:DATA.dates, y:t.y, name:t.name, type:'scatter', mode:'lines',
      line:{color:t.color, width:t.dash==='dash'||t.dash==='dashdot'?1.4:1.8, dash:t.dash},
      connectgaps:false, yaxis:t.y2?'y2':'y',
      hovertemplate:'%{x}<br>%{y}'+(p.unit?' '+p.unit:'')+'<extra>'+t.name+'</extra>'}));
    const lay={height:280, margin:{l:64,r:p.traces.some(t=>t.y2)?64:20,t:8,b:40},
      hovermode:'x unified', showlegend:true,
      legend:{orientation:'h',y:-0.18,x:0,font:{size:11}},
      plot_bgcolor:'#fff', paper_bgcolor:'#fff', xaxis:{gridcolor:'#eef2f7'},
      yaxis:{gridcolor:'#eef2f7',zeroline:true,zerolinecolor:'#94a3b8',zerolinewidth:1,
             title:{text:p.unit||'',font:{size:11}}}};
    if(p.traces.some(t=>t.y2)){const s2=p.traces.find(t=>t.y2);
      lay.yaxis2={overlaying:'y',side:'right',showgrid:false,
        title:{text:s2.name,font:{size:10.5,color:s2.color}}, tickfont:{size:9.5,color:s2.color}};}
    Plotly.newPlot('p'+vi+'_'+i, tr, lay, {responsive:true, displaylogo:false,
      modeBarButtonsToRemove:['lasso2d','select2d','autoScale2d']});
  });
});
const c=v=>`<td class="${v<0?'neg':(v>0?'pos':'')}">${v}</td>`; let prev=null;
document.getElementById('stats').innerHTML=
  '<table><tr><th>合约</th><th>指标</th><th>单位</th><th>样本</th><th>中位数</th>'
  +'<th>均值</th><th>标准差</th><th>最小</th><th>最大</th></tr>'
  + DATA.stats.map(r=>{const sp=(prev&&prev!==r.term)?' class="sep"':''; prev=r.term;
      return `<tr${sp}><td>${r.term}</td><td>${r.label}</td><td>${r.unit}</td><td>${r.n}</td>`
        + `<td class="med ${r.median<0?'neg':(r.median>0?'pos':'')}">${r.median}</td>`
        + c(r.mean) + `<td>${r.std}</td>` + c(r.min) + c(r.max) + '</tr>';}).join('')
  + '</table>';
</script></body></html>"""


_INDEX = """<!DOCTYPE html>
<html lang="zh-CN"><head><meta charset="utf-8"><title>基差分析报告</title>
<style>__CSS__</style></head><body>
<div class="nav">__NAV__</div><h1>股指基差分析报告</h1>
<div class="sub">生成于 __TS__ · 点击上方品种进入分品种报告</div>
<h2>跨品种对比（当月，中位数口径）</h2>
<div class="card">__TABLE__</div></body></html>"""


def _nav(products, cur=None, views=None):
    on = lambda flag: ' class="on"' if flag else ""
    a = [f'<a href="basis_report.html"{on(cur is None)}>总览</a>']
    a += [f'<a href="basis_report_{p}.html"{on(p == cur)}>{PRODUCTS[p]["label"]}</a>' for p in products]
    if views:
        a += [f'<a class="jump" href="#v{i}">{v["label"]}</a>' for i, v in enumerate(views)]
        a += ['<a class="jump" href="#stats">摘要</a>']
    return "".join(a)


def _html_table(df: pd.DataFrame):
    def cell(c):
        if isinstance(c, (int, float)) and not pd.isna(c):
            cls = "neg" if c < 0 else ("pos" if c > 0 else "")
            return f'<td class="{cls}">{c:.2f}</td>'
        return f"<td>{'—' if c is None or (isinstance(c, float) and pd.isna(c)) else c}</td>"
    th = "".join(f"<th>{h}</th>" for h in df.columns)
    body = "".join("<tr>" + "".join(cell(c) for c in r) + "</tr>" for r in df.itertuples(index=False))
    return f"<table><tr>{th}</tr>{body}</table>"


def render_html(results: dict, out_dir=RESULTS_DIR):
    """总览页 + 每品种一页，离线可开（图表库走 CDN）"""
    os.makedirs(out_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y-%m-%d %H:%M")
    valid = {p: d for p, d in results.items() if d is not None and not d.empty}
    if not valid:
        print("✗ 无可导出数据")
        return

    for product, df in valid.items():
        d = df.copy()
        d["date"] = pd.to_datetime(d["date"])
        d = d.sort_values("date")
        views = [{"label": SLOT_LABEL[s], "panels": pans}
                 for s in HTML_SLOTS if (pans := _build(d, panels_for_slot(s)))]
        term = _build(d, panels_term())
        if term:
            views.append({"label": "期限对比", "panels": term})
        if not views:
            continue
        stats = summary_stats(d, HTML_SLOTS)
        stats = [{"term": r["位次"], "label": r["指标"], "unit": r["单位"], "n": int(r["样本"]),
                  "median": r["中位数"], "mean": r["均值"], "std": r["标准差"],
                  "min": r["最小"], "max": r["最大"]} for _, r in stats.iterrows()]
        secs = "".join(f'<h2 id="v{i}">{v["label"]}</h2><div class="card"><div id="chart{i}"></div></div>'
                       for i, v in enumerate(views))
        data = {"dates": d["date"].dt.strftime("%Y-%m-%d").tolist(), "views": views, "stats": stats}
        label = PRODUCTS[product]["label"]
        html = (_PAGE.replace("__CSS__", _CSS).replace("__NAV__", _nav(valid, product, views))
                .replace("__TITLE__", f"{label} 基差分析").replace("__SECTIONS__", secs)
                .replace("__DATA__", json.dumps(data, ensure_ascii=False)).replace("__TS__", ts))
        fp = os.path.join(out_dir, f"basis_report_{product}.html")
        with open(fp, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"  📊 {label}: {len(d)}行, {len(views)}组视图 → {os.path.basename(fp)} "
              f"({os.path.getsize(fp)/1024:.0f} KB)")

    idx = (_INDEX.replace("__CSS__", _CSS).replace("__NAV__", _nav(valid))
           .replace("__TABLE__", _html_table(cross_product_table(valid))).replace("__TS__", ts))
    ip = os.path.join(out_dir, "basis_report.html")
    with open(ip, "w", encoding="utf-8") as f:
        f.write(idx)
    print(f"  ✓ 总览页: {ip}")

In [ ]:
render_html(results)

## 七、静态图（报告用）

与上面交互图同样的列和配色，导出 PNG 到 `results/figs/`。回测散点读 02 写入的 `dividend_backtest`。

In [ ]:
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib import font_manager
_cjk = [f for f in ["Hiragino Sans GB", "PingFang SC", "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Arial Unicode MS"]
        if f in {x.name for x in font_manager.fontManager.ttflist}]
plt.rcParams.update({"font.family": _cjk or ["sans-serif"],
                     "axes.unicode_minus": False, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e5e7eb", "grid.linewidth": 0.6})
FIG_DIR = os.path.join(RESULTS_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)
LS = {"solid": "-", "dash": "--", "dot": ":", "dashdot": "-."}
R = {p: r.set_index("date") for p, r in results.items()}


def plot_traces(ax, df, traces, roll=None):
    for col, name, color, dash in traces:
        if col not in df:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        if roll:
            s = s.rolling(roll, min_periods=max(1, roll // 2)).median()
        ax.plot(s.index, s, LS[dash.split("|")[0]], color=color, lw=1.0, label=name)
    ax.legend(frameon=False, fontsize=8.5, ncol=3, loc="upper left")


def year_axis(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))


def halfyear_axis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7])); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))


def save_fig(fig, name, title, sub):
    h = fig.get_figheight()
    fig.subplots_adjust(top=1 - 0.95 / h)
    fig.text(0.01, 1 - 0.28 / h, title, fontsize=12.5, fontweight="bold", va="bottom")
    fig.text(0.01, 1 - 0.52 / h, sub, fontsize=9, color="#6b7280", va="bottom")
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=170, bbox_inches="tight")
    plt.show()


r3 = R["300"]
# 图1 期货基差：含分红 / 除已实现 / 除预测
p1 = [t for t in panels_for_slot("01")[0][2] if t[0].split("_", 1)[1] in ("basis", "basis_exdiv", "basis_exdiv_pred")]
fig, ax = plt.subplots(figsize=(9, 3.6)); plot_traces(ax, r3.loc["2016":], p1, roll=5)
ax.axhline(0, color="#9ca3af", lw=0.8); ax.set_ylabel("指数点"); year_axis(ax)
save_fig(fig, "fig1_basis_300", "沪深300 次月：分红季期货贴水明显加深，除分红后基差大幅收窄",
         "5 日滚动中位数；实线=期货基差，虚线=除已实现分红，点线=除预测分红")

# 图2 分红：已实现 vs 预测
fig, ax = plt.subplots(figsize=(9, 3.6)); plot_traces(ax, r3.loc["2016":], panels_for_slot("01")[2][2])
ax.set_ylabel("指数点"); year_axis(ax)
save_fig(fig, "fig2_div_300", "沪深300 次月：预测分红紧跟已实现分红，2024 年后下半年误差增大",
         "合约存续期内区间分红点数；误差=预测−已实现")

# 图3 股指期权 − 期货，三个位次
tr = [(f"{s}_iopt_minus_fut", lab, col, "solid") for s, lab, col in
      [("00", "当月", "#2563eb"), ("01", "次月", "#f59e0b"), ("02", "季月1", "#0891b2")]]
fig, ax = plt.subplots(figsize=(9, 3.6)); plot_traces(ax, r3.loc["2022-07":], tr, roll=20)
ax.axhline(0, color="#9ca3af", lw=0.8); ax.set_ylabel("指数点"); halfyear_axis(ax)
save_fig(fig, "fig3_iopt_minus_fut_300", "沪深300：股指期权隐含基差持续高于期货，期限越长利差越大",
         "股指期权隐含基差 − 期货基差，20 日滚动中位数，按到期月对齐")

# 图4 四品种次月年化（除预测分红）
cols = {"50": "#9ca3af", "300": "#2563eb", "500": "#f59e0b", "1000": "#dc2626"}
fig, ax = plt.subplots(figsize=(9, 3.6))
for p, r in R.items():
    s = pd.to_numeric(r.get("01_ann_exdiv_pred"), errors="coerce").loc["2016":].rolling(20, min_periods=10).median()
    ax.plot(s.index, s, color=cols[p], lw=1.1, label=PRODUCTS[p]["label"])
ax.axhline(0, color="#6b7280", lw=0.8); ax.set_ylabel("年化 %"); year_axis(ax)
ax.legend(frameon=False, ncol=4, fontsize=8.5, loc="lower left")
save_fig(fig, "fig4_ann_pred_all", "次月真实升贴水（除预测分红）：中小盘贴水更深，2022 年后中证500/1000 长期年化贴水",
         "年化基差 = (基差 + 预测分红) / 指数 × 365 / 剩余天数，20 日滚动中位数")

# 图5 期货 vs 股指期权 vs ETF 期权
tr = [("01_basis", "期货基差", C_FUT, "solid"), ("01_basis_iopt", "股指期权隐含", C_IOPT, "dashdot"),
      ("01_basis_opt_pred", "ETF期权隐含·减预测分红", C_OPT, "dot")]
fig, ax = plt.subplots(figsize=(9, 3.6)); plot_traces(ax, r3.loc["2022-07":], tr, roll=5)
ax.axhline(0, color="#9ca3af", lw=0.8); ax.set_ylabel("指数点"); halfyear_axis(ax)
save_fig(fig, "fig5_opt_vs_fut_300", "沪深300 次月：两种期权隐含基差大体一致，ETF 版在 ETF 分红公告前偏低",
         "5 日滚动中位数；ETF 版只在公告后（除息前 5 个交易日起）加回 ETF 自身分红")

# 图6 分红预测回测散点（读 02 写入的 dividend_backtest）
bt = pd.read_sql("SELECT * FROM dividend_backtest WHERE product='300'", conn)
fig, axs = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, h in zip(axs, (30, 60, 90)):
    g = bt[bt["horizon"] == h]; m = max(g["real"].max(), g["pred"].max()) * 1.05
    ax.fill_between([0, m], [-3, m - 3], [3, m + 3], color="#dbeafe", lw=0)
    ax.plot([0, m], [0, m], color="#9ca3af", lw=0.8); ax.scatter(g["real"], g["pred"], s=10, color="#2563eb")
    ax.set_title(f"{h} 天窗口  ±3 点内 {(g['err'].abs() <= 3).mean():.0%}", fontsize=10); ax.set_xlabel("实际（股息点指数）")
axs[0].set_ylabel("预测")
save_fig(fig, "fig6_backtest_300", "沪深300 分红点数预测回测：绝大多数样本落在 ±3 点带内",
         f"{bt['asof'].min()[:7]} 至 {bt['asof'].max()[:7]} 每两月一个预测时点，共 {len(bt)} 个样本；蓝带 = 实际 ±3 点")

## 八、口径检查

用合成数据锁住期权到期月对齐、ETF 到期日顺延、股息点归零日等口径，改了上面的计算后跑一遍。

In [ ]:
# 用合成数据锁住几处口径；任何一条不满足都会报 AssertionError
DT, SPOT = pd.Timestamp("2024-11-11"), 4000.0
FUT = {"00": "2411", "01": "2412", "02": "2503", "03": "2506"}           # 期货：当月/次月/两个季月
OPT_MONTHS = ["2411", "2412", "2501", "2503", "2506", "2509"]           # 股指期权：当月 + 下 2 月 + 季月
FWD = {"2411": 3990.0, "2412": 3970.0, "2501": 3950.0, "2503": 3920.0, "2506": 3880.0, "2509": 3850.0}
exp_of = lambda ym: nth_weekday(2000 + int(ym[:2]), int(ym[2:]), 4, 3)


def synthetic_inputs():
    fut = {"date": [DT]}
    for s, ym in FUT.items():
        ltd = exp_of(ym)
        fut.update({f"close_{s}": [FWD[ym]], f"settle_{s}": [np.nan], f"last_trading_date_{s}": [ltd],
                    f"days_{s}": [(ltd - DT).days + 1], f"div_points_{s}": [np.nan], f"contract_code_{s}": [f"IF{ym}"]})
    rows = []
    for ym in OPT_MONTHS:
        for k in np.arange(3800, 4201, 50):          # r=0：PCP 远期 = K + C − P
            c = max(FWD[ym] - k, 0) + 20
            rows.append({"date": DT, "contract_month": ym, "strike": float(k), "expiry": exp_of(ym),
                         "call_close": c, "put_close": c - (FWD[ym] - k), "call_volume": 1, "put_volume": 1,
                         "call_turnover": 1, "put_turnover": 1})
    return (pd.DataFrame({"date": [DT], "index_close": [SPOT]}), pd.DataFrame(fut), pd.DataFrame(),
            pd.DataFrame({"date": [DT], "shibor": [0.0]}), pd.DataFrame(), pd.DataFrame(rows),
            pd.DataFrame(columns=["ex_date", "amount"]))


# 1. 期货合约代码 → 到期年月
assert _fut_month("IF2503") == "2503" and _fut_month(None) is None

# 2. 股指期权按到期月对齐期货：同月 → 隐含基差 = 期货基差、到期日相同
res_t = compute(*synthetic_inputs(), verbose=False).iloc[0]
for s, ym in FUT.items():
    assert res_t[f"{s}_iopt_DTE"] == res_t[f"{s}_DTE"] and abs(res_t[f"{s}_iopt_minus_fut"]) < 1e-6, s

# 3. 没有同月期权的位次留空，不拿别的月份凑
args = list(synthetic_inputs()); args[5] = args[5][args[5]["contract_month"] != "2503"]
res_t = compute(*args, verbose=False).iloc[0]
assert pd.isna(res_t.get("02_basis_iopt")) and abs(res_t["03_iopt_minus_fut"]) < 1e-6

# 4. ETF 期权到期日（第 4 个周三）遇假顺延
cal = pd.DatetimeIndex(pd.bdate_range("2024-10-01", "2024-10-31").drop(pd.Timestamp("2024-10-23")))
assert etf_option_expiry(2024, 10, cal) == pd.Timestamp("2024-10-24")

# 5. 股息点指数归零当天的分红不丢；区间尚未发生则为空
dpi = pd.DataFrame({"date": pd.to_datetime(["2025-12-12", "2025-12-15", "2025-12-16", "2025-12-17"]),
                    "value": [80.0, 81.0, 1.5, 2.0]})
cum = dividend_cum(dpi)
assert abs(_window_div(cum, pd.Timestamp("2025-12-12"), pd.Timestamp("2025-12-17")) - 3.0) < 1e-9
assert np.isnan(_window_div(cum, pd.Timestamp("2025-12-12"), pd.Timestamp("2025-12-31")))

print("✓ 5 项基差口径检查全部通过")